# 05 — Descobrir o que cada componente acrescenta

Há dois componentes que podem alterar uma resposta: o RAG acrescenta evidências recuperadas; o adaptador QLoRA altera o comportamento do gerador. Para separar seus efeitos, comparam-se quatro condições com o mesmo conjunto de perguntas e os mesmos critérios.

```mermaid
graph TD
 A[Qwen base] --> B[sem RAG]
 A --> C[com RAG]
 D[Qwen base + LoRA] --> E[sem RAG]
 D --> F[com RAG]
 B --> G[comparação humana]
 C --> G
 E --> G
 F --> G
```

O teste reservado é usado para comparar, não para escrever exemplos, ajustar hiperparâmetros ou escolher épocas. Assim se reduz o risco de adaptar o experimento às próprias perguntas de avaliação.

In [8]:
# Objetivo: definir diretórios da avaliação.
from pathlib import Path
import json

RANDOM_STATE = 42
DIRETORIO_PROJETO = Path.cwd()
if DIRETORIO_PROJETO.name != "Agente_IA_Transformers_Credito":
    DIRETORIO_PROJETO = DIRETORIO_PROJETO / "Agente_IA_Transformers_Credito"
if not DIRETORIO_PROJETO.is_dir():
    raise FileNotFoundError("Inicie o kernel na raiz do repositório ou na pasta do projeto.")

DIRETORIO_DOCUMENTOS = DIRETORIO_PROJETO / "documentos"
DIRETORIO_PROCESSADOS = DIRETORIO_PROJETO / "dados_processados"
DIRETORIO_RESULTADOS = DIRETORIO_PROJETO / "resultados"
DIRETORIO_PROCESSADOS.mkdir(exist_ok=True)
DIRETORIO_RESULTADOS.mkdir(exist_ok=True)
print("Projeto:", DIRETORIO_PROJETO)

Projeto: d:\Projetos_v2\Agente_IA_Transformers_Credito


## 1. Selecionar as mesmas perguntas para as quatro condições

A comparação usa um arranjo $2\times2$: modelo base ou base + LoRA, cada um sem RAG e com RAG. Para reduzir o custo, serão avaliadas dez perguntas do conjunto de teste. A seleção é determinística e percorre as famílias em rodízio, para evitar que as dez perguntas fiquem concentradas em um único tema.

Cada pergunta será respondida quatro vezes. Assim, serão geradas 40 respostas. O modelo base e o modelo ajustado serão carregados em sequência, liberando a memória da GPU entre eles. As configurações de geração serão mantidas iguais.

O arquivo de revisão reunirá pergunta, resposta de referência, quatro respostas geradas, fontes recuperadas e tempos. A escolha da melhor condição será feita pela leitura das respostas com uma rubrica comum; uma métrica automática servirá como sinal complementar.

In [9]:
# Objetivo: preparar corpus, perguntas de teste e seleção determinàstica de dez exemplos.
import time
import random
import gc
import chromadb
import torch
from sentence_transformers import SentenceTransformer
from rag import ENCODER, COLECAO, carregar_llm, recuperar, montar_mensagens, gerar

# Lê os exemplos e mantém somente os que foram reservados para teste.
exemplos = [
    json.loads(linha)
    for linha in (DIRETORIO_PROJETO / "dataset_sft.jsonl").read_text(encoding="utf-8").splitlines()
    if linha.strip()
]
teste = [exemplo for exemplo in exemplos if exemplo["split"] == "teste"]
N_PERGUNTAS = 10  # Número de perguntas usadas nas quatro condições.
if len(teste) < N_PERGUNTAS:
    raise ValueError(f"São necessárias {N_PERGUNTAS} perguntas de teste; disponíveis: {len(teste)}.")

# Separa por família e embaralha cada grupo com semente fixa para cobrir temas diversos.
grupos_por_familia = {}
gerador_aleatorio = random.Random(RANDOM_STATE)
for exemplo in teste:
    grupos_por_familia.setdefault(exemplo["familia"], []).append(exemplo)
for grupo in grupos_por_familia.values():
    gerador_aleatorio.shuffle(grupo)

# Escolhe uma pergunta por família a cada rodada até completar dez exemplos.
teste_selecionado = []
while len(teste_selecionado) < N_PERGUNTAS:
    houve_selecao = False
    for familia in sorted(grupos_por_familia):
        if grupos_por_familia[familia] and len(teste_selecionado) < N_PERGUNTAS:
            teste_selecionado.append(grupos_por_familia[familia].pop())
            houve_selecao = True
    if not houve_selecao:
        break

# Carrega o encoder na CPU e abre a coleção vetorial persistida do projeto.
encoder = SentenceTransformer(ENCODER, device="cpu")
colecao = chromadb.PersistentClient(
    path=str(DIRETORIO_PROJETO / "banco_chroma")
).get_collection(COLECAO)
CAMINHO_ADAPTADOR = DIRETORIO_PROJETO / "adaptador_pf"
if not CAMINHO_ADAPTADOR.is_dir():
    raise FileNotFoundError("Execute o notebook 04 para criar o adaptador antes de comparar os quatro métodos.")

print(f"Perguntas selecionadas: {len(teste_selecionado)} de {len(teste)}")
for numero, exemplo in enumerate(teste_selecionado, start=1):
    print(f"{numero:02d}. [{exemplo['familia']}] {exemplo['pergunta']}")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9037.39it/s]


Perguntas selecionadas: 10 de 100
01. [avaliacao_governanca] Compare políticas em uma carteira de 49,000 propostas: aprovação 30% e BAD rate aprovado 12%. Quantas aprovações e ocorrências BAD isso representa aproximadamente?
02. [credito_pf] Uma base marca a atraso com data de processamento D+1, depois da concessão em D0. Qual risco deve ser investigado? Explique também como validaria essa conclusão.
03. [qlora_sft] Precisa-se incorporar rapidamente novos procedimentos e apontar fontes. Compare RAG e fine-tuning para esse caso. Diga também o que esse resultado não permite concluir. (formulação 28)
04. [rag_retrieval] Para vetores de consulta [-2, -1, -3] e [5, 3, -2], estime similaridade cosseno e explique por que normalizar não altera sua direção.
05. [transformers] Compare o custo quadrático da atenção para n=32 e n=64. Qual é a razão entre o tamanho das matrizes de atenção? Indique também uma limitação prática. (formulação 6)
06. [avaliacao_governanca] As proporções esperadas em qua

## 2. Executar as quatro condições e salvar material de revisão

Para cada pergunta, o retriever calcula as evidências uma vez; esses mesmos trechos são usados nas duas condições com RAG. Isso evita que uma diferença na busca seja confundida com uma diferença entre modelo base e ajustado. Todas as condições usam o mesmo system prompt; no RAG, são acrescentadas as evidências recuperadas.

São geradas até 256 tokens por resposta para limitar o tempo de inferência. O limite é máximo: a resposta pode terminar antes. O JSON é atualizado após cada geração, permitindo retomar a inspeção mesmo se a célula for interrompida.

Ao final, será criado um Markdown com pergunta, referência, quatro respostas, evidências e campos em branco para avaliação humana. A rubrica separa correção, cobertura, clareza, fidelidade às fontes e invenções. A resposta sem RAG recebe N/A no critério de fidelidade às fontes recuperadas.

In [10]:
# Objetivo: gerar as quatro combinações nas mesmas dez perguntas e salvar cada resultado.
EXECUTAR_AVALIACAO = True  # Mude para False para pular inferência e seguir para métricas de um arquivo já salvo.
MAX_NOVOS_TOKENS = 256  # Teto de saída por resposta; reduzir acelera a execução, mas pode truncar respostas.
CAMINHO_AVALIACAO = DIRETORIO_RESULTADOS / "avaliacao_4_metodos.json"
if EXECUTAR_AVALIACAO:
    avaliacoes = []

    # Recupera uma vez por pergunta; a evidência fica igual para base+RAG e ajustado+RAG.
    evidencias_por_pergunta = {}
    for indice, exemplo in enumerate(teste_selecionado, start=1):
        inicio_busca = time.perf_counter()
        evidencias = recuperar(
            colecao,  # Índice persistente consultado para localizar trechos relevantes.
            encoder,  # Mesmo encoder multilíngue usado para indexar os documentos.
            exemplo["pergunta"],  # Consulta textual usada para pesquisar o corpus.
            usar_mmr=True,  # Seleciona trechos relevantes reduzindo redundância entre eles.
            k=4,  # Quantidade final de trechos que será incluída no prompt.
            fetch_k=12,  # Quantidade de candidatos examinados antes da seleção MMR.
        )
        evidencias_por_pergunta[indice] = {
            "itens": evidencias,
            "tempo_busca_s": time.perf_counter() - inicio_busca,
        }

    # Cada modelo fica na GPU apenas durante seus dois cenários; depois a memória é liberada.
    configuracoes_modelo = [
        ("base", None),
        ("ajustado", CAMINHO_ADAPTADOR),
    ]
    try:
        for nome_modelo, caminho_adaptador in configuracoes_modelo:
            print(f"Carregando modelo: {nome_modelo}")
            modelo, tokenizador = carregar_llm(caminho_adaptador)
            try:
                for indice, exemplo in enumerate(teste_selecionado, start=1):
                    registro_busca = evidencias_por_pergunta[indice]
                    for usar_rag in [False, True]:
                        evidencias = registro_busca["itens"] if usar_rag else []
                        # Mantém o mesmo system prompt; com RAG, só são acrescentadas as evidências.
                        mensagens = montar_mensagens(
                            exemplo["pergunta"],  # Pergunta idêntica nos quatro métodos.
                            evidencias,  # Lista vazia sem RAG; trechos recuperados com RAG.
                        )

                        inicio_geracao = time.perf_counter()
                        resposta = gerar(
                            modelo,  # Modelo base ou modelo base com adaptador LoRA carregado.
                            tokenizador,  # Converte mensagens em tokens e decodifica a saída.
                            mensagens,  # Instruções, pergunta e, se houver, evidências do RAG.
                            max_new_tokens=MAX_NOVOS_TOKENS,  # Limite máximo de tokens gerados.
                        )
                        avaliacao = {
                            "id_pergunta": indice,
                            "familia": exemplo["familia"],
                            "metodo": f"{nome_modelo}_{'com_rag' if usar_rag else 'sem_rag'}",
                            "modelo": nome_modelo,
                            "rag": usar_rag,
                            "pergunta": exemplo["pergunta"],
                            "referencia": exemplo["resposta"],
                            "resposta": resposta,
                            "fontes": evidencias,
                            "tempo_busca_s": registro_busca["tempo_busca_s"] if usar_rag else 0.0,
                            "tempo_geracao_s": time.perf_counter() - inicio_geracao,
                        }
                        avaliacoes.append(avaliacao)

                        # Persiste após cada resposta para reduzir o risco de perder a execução.
                        CAMINHO_AVALIACAO.write_text(
                            json.dumps(avaliacoes, ensure_ascii=False, indent=2),
                            encoding="utf-8",
                        )
                        print(
                            f"{len(avaliacoes):02d}/40 | {avaliacao['metodo']} | "
                            f"pergunta {indice:02d} | {avaliacao['tempo_geracao_s']:.1f} s"
                        )
            finally:
                # Descarta os objetos antes de carregar o próximo modelo e limpa o cache CUDA.
                del modelo, tokenizador
                gc.collect()
                torch.cuda.empty_cache()

    except Exception:
        # O JSON incremental permanece disponível para inspecionar o ponto de falha.
        print(f"Execução interrompida. Respostas já salvas: {len(avaliacoes)} em {CAMINHO_AVALIACAO}")
        raise

    print(f"Avaliação concluída: {len(avaliacoes)} respostas salvas em {CAMINHO_AVALIACAO}")

Carregando modelo: base


Loading weights: 100%|██████████| 434/434 [00:02<00:00, 176.51it/s]


01/40 | base_sem_rag | pergunta 01 | 16.6 s
02/40 | base_com_rag | pergunta 01 | 16.7 s
03/40 | base_sem_rag | pergunta 02 | 16.5 s
04/40 | base_com_rag | pergunta 02 | 16.8 s
05/40 | base_sem_rag | pergunta 03 | 16.5 s
06/40 | base_com_rag | pergunta 03 | 16.7 s
07/40 | base_sem_rag | pergunta 04 | 16.5 s
08/40 | base_com_rag | pergunta 04 | 16.5 s
09/40 | base_sem_rag | pergunta 05 | 16.5 s
10/40 | base_com_rag | pergunta 05 | 16.5 s
11/40 | base_sem_rag | pergunta 06 | 16.4 s
12/40 | base_com_rag | pergunta 06 | 16.5 s
13/40 | base_sem_rag | pergunta 07 | 16.5 s
14/40 | base_com_rag | pergunta 07 | 14.9 s
15/40 | base_sem_rag | pergunta 08 | 16.8 s
16/40 | base_com_rag | pergunta 08 | 16.7 s
17/40 | base_sem_rag | pergunta 09 | 16.5 s
18/40 | base_com_rag | pergunta 09 | 16.7 s
19/40 | base_sem_rag | pergunta 10 | 13.2 s
20/40 | base_com_rag | pergunta 10 | 16.5 s
Carregando modelo: ajustado


Loading weights: 100%|██████████| 434/434 [00:02<00:00, 176.55it/s]


21/40 | ajustado_sem_rag | pergunta 01 | 9.6 s
22/40 | ajustado_com_rag | pergunta 01 | 16.8 s
23/40 | ajustado_sem_rag | pergunta 02 | 10.2 s
24/40 | ajustado_com_rag | pergunta 02 | 8.9 s
25/40 | ajustado_sem_rag | pergunta 03 | 6.6 s
26/40 | ajustado_com_rag | pergunta 03 | 10.0 s
27/40 | ajustado_sem_rag | pergunta 04 | 9.3 s
28/40 | ajustado_com_rag | pergunta 04 | 11.0 s
29/40 | ajustado_sem_rag | pergunta 05 | 10.6 s
30/40 | ajustado_com_rag | pergunta 05 | 13.9 s
31/40 | ajustado_sem_rag | pergunta 06 | 14.7 s
32/40 | ajustado_com_rag | pergunta 06 | 10.7 s
33/40 | ajustado_sem_rag | pergunta 07 | 8.9 s
34/40 | ajustado_com_rag | pergunta 07 | 18.2 s
35/40 | ajustado_sem_rag | pergunta 08 | 14.9 s
36/40 | ajustado_com_rag | pergunta 08 | 7.9 s
37/40 | ajustado_sem_rag | pergunta 09 | 8.0 s
38/40 | ajustado_com_rag | pergunta 09 | 8.8 s
39/40 | ajustado_sem_rag | pergunta 10 | 11.0 s
40/40 | ajustado_com_rag | pergunta 10 | 11.4 s
Avaliação concluída: 40 respostas salvas em d:\P

## 3. Medir similaridade com cosseno e BERTScore

Serão calculadas duas métricas complementares para cada resposta, sempre comparando a saída do modelo com a resposta de referência. A primeira é a similaridade de cosseno entre os embeddings de sentença do encoder usado no RAG. Como os vetores são normalizados, calcula-se o produto escalar: $s = e_{gerada}^{\top} e_{referência}$. Essa medida é rápida e indica a proximidade semântica no espaço do encoder de recuperação.

A segunda métrica é o **BERTScore**, que compara tokens por meio de representações contextuais de um modelo multilíngue (`bert-base-multilingual-cased`). Para cada token gerado, procura-se o token semanticamente mais próximo na referência; essa correspondência contribui para a **precisão** (*precision*). No sentido inverso, cada token da referência procura uma correspondência na resposta gerada; isso contribui para a **revocação** (*recall*). O F1 combina as duas medidas pela média harmônica:

$$F_1 = 2 \cdot \frac{P \cdot R}{P + R}$$

O F1 será usado como resumo principal, enquanto precisão e revocação ajudam a interpretar os resultados. Uma resposta extensa pode cobrir mais conteúdo da referência e ter revocação maior, mas também incluir conteúdo irrelevante e ter precisão menor. Nenhuma dessas métricas comprova factualidade, cobertura adequada ou fidelidade às fontes; os escores serão analisados junto com a revisão humana.

As duas métricas são obrigatórias e serão calculadas para as mesmas 40 respostas: dez perguntas em quatro condições. O cosseno usa o encoder já carregado para o RAG. O BERTScore carrega seu próprio modelo multilíngue em CPU, com lotes pequenos para não disputar memória com a GPU usada pelo Qwen. Na primeira execução, o modelo BERT pode precisar ser baixado; depois, ficará no cache para as próximas execuções.

Serão salvos, por resposta e como média por método, os escores de precisão, revocação e F1 do BERTScore, além da similaridade de cosseno. Se o pacote ou o modelo não estiver disponível, a célula interromperá com erro, sem substituir silenciosamente a métrica.

In [11]:
# Objetivo: calcular cosseno e BERTScore e montar o material de revisão humana.
from collections import defaultdict
import csv
import statistics
import numpy as np
from bert_score import score as calcular_bertscore

# Lê as respostas salvas; não ? necessário carregar novamente o Qwen nesta etapa.
if not CAMINHO_AVALIACAO.is_file():
    raise FileNotFoundError("Execute a célula de geração antes de calcular as métricas.")
avaliacoes = json.loads(CAMINHO_AVALIACAO.read_text(encoding="utf-8"))
if len(avaliacoes) != N_PERGUNTAS * 4:
    raise ValueError(f"Esperadas {N_PERGUNTAS * 4} respostas; o arquivo contém {len(avaliacoes)}.")
if "encoder" not in globals():
    raise RuntimeError("Encoder não carregado. Execute a célula de configuração do corpus antes desta avaliação.")

# Codifica respostas e referências com o encoder multilíngue já usado na recuperação.
textos_para_comparar = []
for registro in avaliacoes:
    textos_para_comparar.extend([registro["resposta"], registro["referencia"]])
embeddings = encoder.encode(
    textos_para_comparar,  # Pares alternados: resposta gerada e referência correspondente.
    normalize_embeddings=True,  # Normaliza cada vetor para obter o cosseno pelo produto escalar.
    batch_size=16,  # Número de textos processados por lote pelo encoder.
    show_progress_bar=True,  # Exibe o andamento da codificação.
)

# Para vetores unitários, cos(a,b) = a^T b; registra um escore por resposta.
for indice, registro in enumerate(avaliacoes):
    vetor_gerado = embeddings[2 * indice]
    vetor_referencia = embeddings[2 * indice + 1]
    escore = float(np.dot(vetor_gerado, vetor_referencia))
    if not np.isfinite(escore):
        raise ValueError(f"Similaridade não finita na pergunta {registro['id_pergunta']}.")
    registro["similaridade_cosseno"] = escore

# BERTScore compara tokens com embeddings contextuais; CPU evita competir com o Qwen na GPU.
respostas_geradas = [registro["resposta"] for registro in avaliacoes]
respostas_referencia = [registro["referencia"] for registro in avaliacoes]
precisao_bert, recall_bert, f1_bert = calcular_bertscore(
    cands=respostas_geradas,  # Respostas produzidas pelos quatro métodos.
    refs=respostas_referencia,  # Respostas de referência correspondentes, na mesma ordem.
    model_type="bert-base-multilingual-cased",  # Encoder multilíngue adequado a textos em português.
    lang="pt",  # Idioma dos textos comparados; usado na configuração/identificação da métrica.
    device="cpu",  # Mantém a memória da GPU disponível para os modelos generativos.
    batch_size=8,  # Limita memória RAM usada durante a inferência do BERTScore.
    verbose=True,  # Exibe o progresso do cálculo.
    rescale_with_baseline=False,  # Mantém os escores brutos, sem baseline específico de idioma.
)

# Associa precision, recall e F1 a cada resposta; os três valores são preservados.
for indice, registro in enumerate(avaliacoes):
    registro["bertscore_precision"] = float(precisao_bert[indice].item())
    registro["bertscore_recall"] = float(recall_bert[indice].item())
    registro["bertscore_f1"] = float(f1_bert[indice].item())
    escores_bert = [
        registro["bertscore_precision"],
        registro["bertscore_recall"],
        registro["bertscore_f1"],
    ]
    if not all(np.isfinite(escore) for escore in escores_bert):
        raise ValueError(f"BERTScore não finito na pergunta {registro['id_pergunta']}.")

# Resume métricas e tempo por método, mantendo os escores individuais nos arquivos de saída.
ordem_metodos = ["base_sem_rag", "base_com_rag", "ajustado_sem_rag", "ajustado_com_rag"]
resumo_metricas = {}
for metodo in ordem_metodos:
    registros_metodo = [registro for registro in avaliacoes if registro["metodo"] == metodo]
    if len(registros_metodo) != N_PERGUNTAS:
        raise ValueError(f"Método {metodo}: esperadas {N_PERGUNTAS} respostas, encontradas {len(registros_metodo)}.")
    resumo_metricas[metodo] = {
        "similaridade_cosseno_media": statistics.mean(
            registro["similaridade_cosseno"] for registro in registros_metodo
        ),
        "bertscore_precision_media": statistics.mean(
            registro["bertscore_precision"] for registro in registros_metodo
        ),
        "bertscore_recall_media": statistics.mean(
            registro["bertscore_recall"] for registro in registros_metodo
        ),
        "bertscore_f1_medio": statistics.mean(
            registro["bertscore_f1"] for registro in registros_metodo
        ),
        "tempo_geracao_medio_s": statistics.mean(
            registro["tempo_geracao_s"] for registro in registros_metodo
        ),
        "n_perguntas": len(registros_metodo),
    }

# Agrupa resultados para apresentar pergunta e as quatro respostas lado a lado.
por_pergunta = defaultdict(dict)
for registro in avaliacoes:
    por_pergunta[registro["id_pergunta"]][registro["metodo"]] = registro
nomes_metodos = {
    "base_sem_rag": "Base — sem RAG",
    "base_com_rag": "Base — com RAG",
    "ajustado_sem_rag": "Base + LoRA — sem RAG",
    "ajustado_com_rag": "Base + LoRA — com RAG",
}
CAMINHO_REVISAO = DIRETORIO_RESULTADOS / "revisao_qualitativa_4_metodos.md"
linhas_markdown = [
    "# Revisão qualitativa: quatro métodos",
    "",
    "Avaliar cada resposta de 1 a 5 em correção, cobertura e clareza. Para métodos com RAG, avaliar também fidelidade às fontes; para métodos sem RAG, marcar N/A. Registrar se houve informação inventada.",
    "",
]
for indice in sorted(por_pergunta):
    registros = por_pergunta[indice]
    if any(metodo not in registros for metodo in ordem_metodos):
        raise ValueError(f"A pergunta {indice} não possui respostas nos quatro métodos.")
    referencia = registros[ordem_metodos[0]]
    linhas_markdown.extend([
        f"## Pergunta {indice:02d} — {referencia['familia']}",
        "",
        f"**Pergunta:** {referencia['pergunta']}",
        "",
        f"**Resposta de referência:** {referencia['referencia']}",
        "",
    ])
    for metodo in ordem_metodos:
        registro = registros[metodo]
        linhas_markdown.extend([
            f"### {nomes_metodos[metodo]}",
            "",
            registro["resposta"],
            "",
            (
                f"Cosseno: {registro['similaridade_cosseno']:.4f} | "
                f"BERTScore P/R/F1: {registro['bertscore_precision']:.4f} / "
                f"{registro['bertscore_recall']:.4f} / {registro['bertscore_f1']:.4f} | "
                f"Tempo: {registro['tempo_geracao_s']:.1f} s"
            ),
            "",
        ])
        if registro["rag"]:
            linhas_markdown.append("**Trechos recuperados:**")
            for fonte in registro["fontes"]:
                linhas_markdown.append(
                    f"- [{fonte.get('source_id', 'fonte')}, p. {fonte.get('page', '?')}] {fonte.get('texto', '')}"
                )
            linhas_markdown.append("")
        linhas_markdown.extend([
            "**Avaliação humana:** correção __/5 | cobertura __/5 | clareza __/5 | fidelidade às fontes __/5 ou N/A | inventou informação? __ | observações: __",
            "",
        ])
CAMINHO_REVISAO.write_text("\n".join(linhas_markdown), encoding="utf-8")

# Salva métricas por resposta em CSV e médias por método em JSON.
CAMINHO_METRICAS = DIRETORIO_RESULTADOS / "metricas_4_metodos.csv"
campos_csv = [
    "id_pergunta", "familia", "metodo", "similaridade_cosseno",
    "bertscore_precision", "bertscore_recall", "bertscore_f1", "tempo_geracao_s",
]
with CAMINHO_METRICAS.open("w", newline="", encoding="utf-8-sig") as arquivo_csv:
    escritor = csv.DictWriter(arquivo_csv, fieldnames=campos_csv)
    escritor.writeheader()
    escritor.writerows(
        {campo: registro.get(campo) for campo in campos_csv}
        for registro in avaliacoes
    )
CAMINHO_RESUMO = DIRETORIO_RESULTADOS / "resumo_metricas_4_metodos.json"
CAMINHO_RESUMO.write_text(
    json.dumps(resumo_metricas, ensure_ascii=False, indent=2), encoding="utf-8"
)
CAMINHO_AVALIACAO.write_text(
    json.dumps(avaliacoes, ensure_ascii=False, indent=2), encoding="utf-8"
)

print(json.dumps(resumo_metricas, ensure_ascii=False, indent=2))
print("Documento para revisão humana:", CAMINHO_REVISAO)
print("Métricas individuais:", CAMINHO_METRICAS)

Batches: 100%|██████████| 5/5 [00:01<00:00,  2.72it/s]
c:\Users\leova\anaconda3\envs\engenharia_ia_venv\lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\leova\.cache\huggingface\hub\models--bert-base-multilingual-cased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 199

calculating scores...
computing bert embedding.


100%|██████████| 7/7 [00:05<00:00,  1.36it/s]


computing greedy matching.


100%|██████████| 5/5 [00:00<00:00, 121.84it/s]

done in 5.18 seconds, 7.72 sentences/sec
{
  "base_sem_rag": {
    "similaridade_cosseno_media": 0.6532193094491958,
    "bertscore_precision_media": 0.654954069852829,
    "bertscore_recall_media": 0.6893842101097107,
    "bertscore_f1_medio": 0.6716991662979126,
    "tempo_geracao_medio_s": 16.19728618999907,
    "n_perguntas": 10
  },
  "base_com_rag": {
    "similaridade_cosseno_media": 0.5714873760938645,
    "bertscore_precision_media": 0.6428285658359527,
    "bertscore_recall_media": 0.6840425670146942,
    "bertscore_f1_medio": 0.662594485282898,
    "tempo_geracao_medio_s": 16.455933999999615,
    "n_perguntas": 10
  },
  "ajustado_sem_rag": {
    "similaridade_cosseno_media": 0.7209590822458267,
    "bertscore_precision_media": 0.7228332102298737,
    "bertscore_recall_media": 0.730914568901062,
    "bertscore_f1_medio": 0.7267195284366608,
    "tempo_geracao_medio_s": 10.39474959999934,
    "n_perguntas": 10
  },
  "ajustado_com_rag": {
    "similaridade_cosseno_media": 0.7

## 4. Conclusão

Neste teste exploratório com dez perguntas, o ajuste QLoRA elevou o BERTScore F1 médio de 0,672 para 0,727 sem RAG e de 0,663 para 0,719 com RAG. Já o RAG não aumentou a média em relação ao cenário sem recuperação: o F1 caiu de 0,672 para 0,663 no modelo base e de 0,727 para 0,719 no ajustado.

Os resultados sugerem que o ajuste aproximou as respostas das referências neste conjunto, mas não comprovam correção factual nem generalização. O RAG ainda pode ser útil quando recupera evidências pontuais e pertinentes; essa hipótese deve ser verificada pela revisão das fontes e respostas e por uma avaliação com mais perguntas.